# LunaProof — real-DEM, ground-truth benchmark (SIH26166 · Team Maximus2 · ID 185903)

**Run order: Runtime ▸ Change runtime type ▸ T4 GPU, then Run all. Total ≈ 35–50 min. Results are cached, so a disconnect can be resumed by re-running.**

What this notebook does, and what it does **not** claim
| | |
|---|---|
| **Real data** | NASA LRO **LOLA LDEM_64** elevation (real lunar topography, ~474 m/px). Not synthetic terrain. |
| **Physics** | Lommel–Seeliger shading + true **cast shadows** at any sun azimuth/elevation. No atmosphere ⇒ no fill light. |
| **Ground truth** | Exact by construction (known rotation/scale). Error is measured at **held-out checkpoints**, never on the fitted matches. |
| **No leakage** | Train / validation / test are **different lunar regions**; a code assertion checks the tiles do not overlap. |
| **Methods** | SIFT · phase-congruency+SIFT · LoFTR (pretrained, zero-shot) · phase-congruency detector + **learned descriptor trained here** |
| **Not claimed** | This is not Chandrayaan-2 data. ~474 m/px is far coarser than OHRC (0.25 m). Results show *illumination behaviour*, not final ISRO accuracy. SPICE/PDS4 ingestion is future work. |

In [ ]:
# 1) Setup ---------------------------------------------------------------
!pip -q install kornia tabulate
import os, time, json, math, random, zipfile, warnings
import numpy as np, cv2, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False
DEV = 'cuda' if (HAVE_TORCH and torch.cuda.is_available()) else 'cpu'
RES = os.environ.get('LP_RES', '/content/lunaproof_results'); os.makedirs(RES, exist_ok=True)
USE_SYNTHETIC_DEBUG = os.environ.get('LP_DEBUG_SYNTH', '0') == '1'   # NEVER True for real results
QUICK = os.environ.get('LP_QUICK', '0') == '1'                       # True = tiny run just to test the code
SEED = 7; random.seed(SEED); np.random.seed(SEED)
import zlib
def stable_seed(*a): return zlib.crc32('|'.join(map(str, a)).encode()) % (2**32)
if HAVE_TORCH: torch.manual_seed(SEED)
print('device:', DEV, '| cv2', cv2.__version__, '| torch', torch.__version__ if HAVE_TORCH else None)
assert DEV == 'cuda' or QUICK or not HAVE_TORCH, 'Switch runtime to T4 GPU (Runtime > Change runtime type)'
if USE_SYNTHETIC_DEBUG: print('!!! SYNTHETIC DEBUG MODE - results are NOT real-terrain results !!!')

In [ ]:
# 2) Physics renderer, Kovesi log-Gabor phase congruency, matchers, ground-truth metrics
import numpy as np, cv2, math

# ---------------------------------------------------------------- physics renderer
def render_lunar(dem_m, px_x_m, px_y_m, az_deg, el_deg, noise=0.01, psf_sigma=0.8,
                 max_steps=160, seed=0):
    """Lommel-Seeliger shading + real cast shadows of a DEM. Compass azimuth (0=N, 90=E), image is north-up.
    Nadir viewing (e = slope angle). No atmosphere => no fill light. Returns float image in [0,1] and shadow mask."""
    rng = np.random.default_rng(seed)
    dem = dem_m.astype(np.float32)
    H, W = dem.shape
    dz_drow, dz_dcol = np.gradient(dem, px_y_m, px_x_m)          # metres / metre
    gx, gn = dz_dcol, -dz_drow                                   # east, north gradients
    nz = 1.0 / np.sqrt(1 + gx**2 + gn**2)
    nx, ny = -gx * nz, -gn * nz
    az, el = math.radians(az_deg), math.radians(el_deg)
    L = np.array([math.cos(el) * math.sin(az), math.cos(el) * math.cos(az), math.sin(el)], np.float32)
    mu0 = np.clip(nx * L[0] + ny * L[1] + nz * L[2], 0, None)    # cos(incidence)
    mu = nz                                                      # cos(emission), nadir camera
    ls = mu0 / (mu0 + mu + 1e-6)                                 # Lommel-Seeliger
    # cast shadows: march toward the sun, compare terrain height to the sun ray
    dcol, drow = math.sin(az), -math.cos(az)
    step_m = math.hypot(dcol * px_x_m, drow * px_y_m)
    rr, cc = np.mgrid[0:H, 0:W].astype(np.float32)
    shadow = np.zeros((H, W), bool)
    tan_el = math.tan(el)
    for s in range(1, max_steps + 1):
        mapx, mapy = cc + s * dcol, rr + s * drow
        h = cv2.remap(dem, mapx, mapy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=-1e9)
        shadow |= (h - dem) > s * step_m * tan_el
    img = ls * (~shadow)
    img = img * (0.6 + 0.4 * math.sin(el))                       # brightness depends on sun elevation
    img = cv2.GaussianBlur(img, (0, 0), psf_sigma)               # optics PSF
    img = img + rng.normal(0, noise, img.shape).astype(np.float32)
    lo, hi = np.percentile(img, [1, 99.5])
    img = np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
    return img.astype(np.float32), shadow

def to_u8(x): return np.clip(x * 255, 0, 255).astype(np.uint8)

# ---------------------------------------------------------------- Kovesi log-Gabor phase congruency
def phase_congruency(img_u8, nscale=4, norient=6, min_wl=8.0, mult=2.1, sigma_onf=0.55,
                     dtheta_on_sigma=1.3, k=2.0, cutoff=0.5, gain=10.0, pad=32):
    f = img_u8.astype(np.float32) / 255.0
    f = np.pad(f, pad, mode='reflect')
    H, W = f.shape
    fy = np.fft.fftfreq(H)[:, None]; fx = np.fft.fftfreq(W)[None, :]
    radius = np.sqrt(fx**2 + fy**2); radius[0, 0] = 1.0
    theta = np.arctan2(-fy, fx)
    sint, cost = np.sin(theta), np.cos(theta)
    lowpass = 1.0 / (1.0 + (radius / 0.45) ** 30)
    F = np.fft.fft2(f)
    log_gabor = []
    for s in range(nscale):
        fo = 1.0 / (min_wl * mult ** s)
        lg = np.exp(-(np.log(radius / fo)) ** 2 / (2 * np.log(sigma_onf) ** 2)) * lowpass
        lg[0, 0] = 0
        log_gabor.append(lg)
    theta_sigma = math.pi / norient / dtheta_on_sigma
    pc_sum = np.zeros((H, W), np.float32)
    covx2 = np.zeros_like(pc_sum); covy2 = np.zeros_like(pc_sum); covxy = np.zeros_like(pc_sum)
    for o in range(norient):
        ang = o * math.pi / norient
        ds = sint * math.cos(ang) - cost * math.sin(ang)
        dc = cost * math.cos(ang) + sint * math.sin(ang)
        spread = np.exp(-np.arctan2(ds, dc) ** 2 / (2 * theta_sigma ** 2))
        sumE = np.zeros((H, W), np.float32); sumO = np.zeros_like(sumE); sumAn = np.zeros_like(sumE)
        maxAn = np.zeros_like(sumE); EOs = []; tau = None
        for s in range(nscale):
            EO = np.fft.ifft2(F * log_gabor[s] * spread)
            An = np.abs(EO).astype(np.float32)
            sumAn += An; sumE += EO.real.astype(np.float32); sumO += EO.imag.astype(np.float32)
            maxAn = np.maximum(maxAn, An)
            if s == 0: tau = np.median(An) / math.sqrt(math.log(4))
            EOs.append(EO)
        xen = np.sqrt(sumE**2 + sumO**2) + 1e-4
        mE, mO = sumE / xen, sumO / xen
        energy = np.zeros((H, W), np.float32)
        for EO in EOs:
            energy += (EO.real * mE + EO.imag * mO).astype(np.float32) - np.abs(EO.real * mO - EO.imag * mE).astype(np.float32)
        T = tau * math.sqrt(math.pi / 2) + k * tau * math.sqrt((4 - math.pi) / 2)
        energy = np.maximum(energy - T, 0)
        width = (sumAn / (maxAn + 1e-4) - 1) / (nscale - 1)
        weight = 1.0 / (1.0 + np.exp((cutoff - width) * gain))
        pc = weight * energy / (sumAn + 1e-4)
        pc_sum += pc
        covx2 += (pc * math.cos(ang)) ** 2; covy2 += (pc * math.sin(ang)) ** 2; covxy += (pc * math.cos(ang)) * (pc * math.sin(ang))
    a, b, c = covx2, covxy, covy2
    M = 0.5 * (a + c + np.sqrt(4 * b**2 + (a - c) ** 2))       # max moment: edge-ness
    sl = (slice(pad, H - pad), slice(pad, W - pad))
    return (pc_sum / norient)[sl], M[sl]

def pc_to_u8(pc, blur=2.0):
    pc = cv2.GaussianBlur(pc.astype(np.float32), (0, 0), blur)
    hi = np.percentile(pc, 99.5) + 1e-9
    return np.clip(pc / hi * 255, 0, 255).astype(np.uint8)

# ---------------------------------------------------------------- pair generation with exact ground truth
def make_pair(dem, px_x, px_y, az_ref, el_ref, az_src, el_src, rot_deg, scale, seed=0):
    ref, _ = render_lunar(dem, px_x, px_y, az_ref, el_ref, seed=seed)
    srcd, _ = render_lunar(dem, px_x, px_y, az_src, el_src, seed=seed + 1)
    H, W = ref.shape
    M = cv2.getRotationMatrix2D((W / 2, H / 2), rot_deg, scale).astype(np.float64)  # p_src = M @ [p_ref,1]
    src = cv2.warpAffine(srcd, M, (W, H), flags=cv2.INTER_LINEAR)
    valid_src = cv2.warpAffine(np.ones_like(srcd), M, (W, H), flags=cv2.INTER_NEAREST) > 0.5
    valid_src = cv2.erode(valid_src.astype(np.uint8), np.ones((3, 3), np.uint8), iterations=24).astype(bool)
    return to_u8(ref), to_u8(src), M, valid_src

# ---------------------------------------------------------------- matchers (all return matched point arrays)
def mutual_ratio_match(d1, d2, ratio=0.85):
    """brute-force L2 matcher with ratio test + mutual check; works for SIFT float descriptors"""
    if d1 is None or d2 is None or len(d1) < 2 or len(d2) < 2: return np.zeros((0, 2), int)
    bf = cv2.BFMatcher(cv2.NORM_L2)
    m12 = bf.knnMatch(d1, d2, k=2); m21 = bf.match(d2, d1)
    back = {m.queryIdx: m.trainIdx for m in m21}
    out = []
    for pair in m12:
        if len(pair) < 2: continue
        a, b = pair
        if a.distance < ratio * b.distance and back.get(a.trainIdx, -1) == a.queryIdx:
            out.append((a.queryIdx, a.trainIdx))
    return np.array(out, int).reshape(-1, 2)

def sift_match(a_u8, b_u8, mask_b=None, nfeat=4000, ratio=0.85, contrast=0.01):
    sift = cv2.SIFT_create(nfeatures=nfeat, contrastThreshold=contrast)
    k1, d1 = sift.detectAndCompute(a_u8, None)
    k2, d2 = sift.detectAndCompute(b_u8, (mask_b.astype(np.uint8) * 255) if mask_b is not None else None)
    idx = mutual_ratio_match(d1, d2, ratio)
    if len(idx) == 0: return np.zeros((0, 2), np.float32), np.zeros((0, 2), np.float32)
    return (np.float32([k1[i].pt for i in idx[:, 0]]), np.float32([k2[j].pt for j in idx[:, 1]]))

def sift_on_pc(a_u8, b_u8, mask_b=None, **kw):
    pa, _ = phase_congruency(a_u8); pb, _ = phase_congruency(b_u8)
    return sift_match(pc_to_u8(pa), pc_to_u8(pb), mask_b, **kw)

# ---------------------------------------------------------------- verification + metrology
def verify(pa, pb, thr=3.0):
    if len(pa) < 4: return None, np.zeros(len(pa), bool)
    H, m = cv2.findHomography(pa.reshape(-1, 1, 2), pb.reshape(-1, 1, 2), cv2.USAC_MAGSAC, thr, maxIters=5000, confidence=0.999)
    if H is None or m is None: return None, np.zeros(len(pa), bool)
    return H, m.ravel().astype(bool)

def apply_H(H, pts):
    p = np.c_[pts, np.ones(len(pts))] @ H.T
    return p[:, :2] / p[:, 2:3]

def evaluate(pa, pb, M_gt, valid_src, gsd_m, size, grid=8):
    """Ground-truth evaluation: error at held-out CHECKPOINTS (not the fitted matches)."""
    res = dict(n_matches=len(pa), n_inliers=0, inlier_ratio=0.0, match_precision=0.0,
               med_err_px=np.inf, rmse_px=np.inf, p95_px=np.inf, pct_lt2px=0.0, coverage=0.0, med_err_m=np.inf)
    if len(pa) == 0: return res
    # independent of the estimator: are the raw matches correct w.r.t. ground truth?
    gt_b = (np.c_[pa, np.ones(len(pa))] @ M_gt.T)
    res['match_precision'] = float(np.mean(np.linalg.norm(gt_b - pb, axis=1) < 3.0))
    H, inl = verify(pa, pb)
    res['n_inliers'] = int(inl.sum()); res['inlier_ratio'] = float(inl.mean()) if len(inl) else 0.0
    if H is None: return res
    cells = set((int(x * grid / size), int(y * grid / size)) for x, y in pa[inl]); res['coverage'] = len(cells) / grid**2
    g = np.linspace(40, size - 40, 20); gx, gy = np.meshgrid(g, g)
    chk = np.c_[gx.ravel(), gy.ravel()].astype(np.float64)
    gt = np.c_[chk, np.ones(len(chk))] @ M_gt.T
    ok = (gt[:, 0] > 0) & (gt[:, 0] < size) & (gt[:, 1] > 0) & (gt[:, 1] < size)
    ok &= valid_src[np.clip(gt[:, 1].astype(int), 0, size - 1), np.clip(gt[:, 0].astype(int), 0, size - 1)]
    est = apply_H(H, chk)
    err = np.linalg.norm(est[ok] - gt[ok], axis=1)
    if len(err) == 0 or not np.isfinite(err).all(): return res
    res.update(med_err_px=float(np.median(err)), rmse_px=float(np.sqrt(np.mean(err**2))), p95_px=float(np.percentile(err, 95)),
               pct_lt2px=float(np.mean(err < 2.0)), med_err_m=float(np.median(err) * gsd_m))
    return res

def gate(res):
    """Refusal gate uses ONLY quantities observable without ground truth."""
    if res['n_inliers'] < 15 or res['inlier_ratio'] < 0.15 or res['coverage'] < 0.15: return 'REFUSED'
    if res['n_inliers'] < 40 or res['coverage'] < 0.30: return 'DEGRADED'
    return 'SUCCESS'


In [ ]:
# 3) LOLA DEM reader (HTTP Range request: downloads only the rows we need)
import numpy as np, os, math, requests
MOON_R = 1737400.0
M_PER_DEG = 2 * math.pi * MOON_R / 360.0

def read_range(src, start, length):
    if os.path.exists(src):
        with open(src, 'rb') as f:
            f.seek(start); return f.read(length)
    r = requests.get(src, headers={'Range': f'bytes={start}-{start+length-1}'}, timeout=180)
    if r.status_code != 206:
        raise RuntimeError(f'server did not honour Range request (HTTP {r.status_code}) for {src}')
    return r.content

def fetch_window(src, lat, lon_e, size, ppd=64, W=23040, H=11520, scale=0.5):
    """LOLA LDEM cylindrical GDR: row=(90-lat)*ppd, col=lon_east*ppd, int16 DN*0.5 = metres. Returns dem(m), px_x_m, px_y_m."""
    r0 = int(round((90 - lat) * ppd - size / 2)); r0 = max(0, min(H - size, r0))
    c0 = int(round(lon_e * ppd - size / 2))
    raw = read_range(src, r0 * W * 2, size * W * 2)
    dem = None
    for dt in ('<i2', '>i2'):                                  # detect byte order by physical sanity
        a = np.frombuffer(raw, dtype=dt).reshape(size, W)
        cols = np.arange(c0, c0 + size) % W
        z = a[:, cols].astype(np.float32) * scale
        if np.abs(z).max() < 12000 and z.std() > 20:
            dem = z; break
    if dem is None: raise RuntimeError('DEM window failed sanity check (bad byte order / wrong file?)')
    lat_c = 90 - (r0 + size / 2) / ppd
    px_y = M_PER_DEG / ppd
    px_x = px_y * math.cos(math.radians(lat_c))
    return dem, px_x, px_y, lat_c

def boxes_overlap(a, b, size_deg):
    dlat = abs(a[0] - b[0]); dlon = abs(a[1] - b[1]); dlon = min(dlon, 360 - dlon)
    return dlat < size_deg and dlon < size_deg


In [ ]:
# 4) Real lunar tiles, split BY REGION (no leakage) ------------------------
SIZE = 640
PPD = 64
TILES = {   # name: (lat, lon_east, role)
 'serenitatis': (28, 18, 'train'), 'imbrium': (33, 344.4, 'train'), 'fecunditatis': (-8, 51, 'train'),
 'nubium': (-21, 328, 'train'), 'procellarum': (0, 300, 'train'), 'descartes_hl': (-12, 16, 'train'),
 'farside_a': (0, 200, 'train'), 'farside_b': (-30, 140, 'train'), 'hadley': (26, 3, 'train'),
 'tranquillitatis': (8.5, 31.4, 'val'), 'south_highland': (-35, 25, 'val'),
 'tycho': (-43.3, 348.9, 'test'), 'copernicus': (9.6, 339.9, 'test'),
 'aristarchus': (23.7, 312.5, 'test'), 'orientale': (-19, 265, 'test'),
}
if QUICK:
    TILES = {k: v for k, v in TILES.items() if k in ('serenitatis', 'imbrium', 'tranquillitatis', 'tycho', 'copernicus')}
deg = SIZE / PPD
names = list(TILES)
for i in range(len(names)):                      # leakage check: NO two tiles of different roles may overlap
    for j in range(i + 1, len(names)):
        a, b = TILES[names[i]], TILES[names[j]]
        assert not boxes_overlap(a, b, deg + 1), f'tiles overlap: {names[i]} / {names[j]}'
print('leakage check passed:', {r: sum(v[2] == r for v in TILES.values()) for r in ('train', 'val', 'test')})

CANDIDATE_URLS = [
  'https://pds-geosciences.wustl.edu/lro/lro-l-lola-3-rdr-v1/lrolol_1xxx/data/lola_gdr/cylindrical/img/ldem_64.img',
  'https://imbrium.mit.edu/DATA/LOLA_GDR/CYLINDRICAL/IMG/LDEM_64.IMG',
]
LDEM_SRC = None
def pick_source():
    global LDEM_SRC
    if os.path.exists('/content/LDEM_64.IMG'): LDEM_SRC = '/content/LDEM_64.IMG'; return
    for u in CANDIDATE_URLS:
        try:
            fetch_window(u, -43.3, 348.9, 64, ppd=PPD); LDEM_SRC = u; print('DEM source OK (HTTP Range):', u); return
        except Exception as e: print('  source failed:', u, '->', str(e)[:120])
    print('Range fetch unavailable -> trying full download of LDEM_64 (~530 MB)...')
    import requests
    for u in CANDIDATE_URLS:
        try:
            with requests.get(u, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open('/content/LDEM_64.IMG', 'wb') as f:
                    for ch in r.iter_content(1 << 20): f.write(ch)
            LDEM_SRC = '/content/LDEM_64.IMG'; print('downloaded', u); return
        except Exception as e: print('  download failed:', u, '->', str(e)[:120])
    raise RuntimeError('Could not get LOLA LDEM_64. Download LDEM_64.IMG manually from the LOLA GDR page on the '
                       'NASA PDS Geosciences Node, upload it to /content/LDEM_64.IMG, and re-run this cell. '
                       'DO NOT continue with fake terrain.')

def synth_tile(seed):     # ONLY for code debugging in the sandbox
    rng = np.random.default_rng(seed); N = SIZE; z = np.zeros((N, N), np.float32)
    for o in range(6):
        f = 2 ** o; g = rng.normal(size=(f * 4 + 1, f * 4 + 1)).astype(np.float32)
        z += cv2.resize(g, (N, N), interpolation=cv2.INTER_CUBIC) * (600 / f)
    yy, xx = np.mgrid[0:N, 0:N]
    for _ in range(50):
        cx, cy = rng.integers(0, N, 2); R = rng.uniform(12, 60); d = R * 0.2 * 470; r = np.hypot(xx - cx, yy - cy) / R
        z += np.where(r < 1, -d * (1 - r**2), 0) + np.where((r >= 1) & (r < 1.4), 0.35 * d * np.exp(-((r - 1.15) / 0.12) ** 2), 0)
    return z, 470.0, 470.0, 0.0

DEMS = {}
if USE_SYNTHETIC_DEBUG:
    for i, k in enumerate(TILES): DEMS[k] = synth_tile(i)
else:
    pick_source()
    for k, (lat, lon, role) in TILES.items():
        DEMS[k] = fetch_window(LDEM_SRC, lat, lon, SIZE, ppd=PPD)
        z = DEMS[k][0]; print(f'{k:16s} {role:5s} lat {DEMS[k][3]:6.1f}  relief {z.max()-z.min():7.0f} m  px {DEMS[k][1]:.0f}x{DEMS[k][2]:.0f} m')
fig, ax = plt.subplots(1, min(5, len(DEMS)), figsize=(16, 3.4))
for a, k in zip(np.atleast_1d(ax), list(DEMS)[-5:]):
    a.imshow(DEMS[k][0], cmap='terrain'); a.set_title(k + ' (' + TILES[k][2] + ')', fontsize=9); a.axis('off')
plt.tight_layout(); plt.savefig(f'{RES}/fig0_real_dem_tiles.png', dpi=150); plt.show()

In [ ]:
# 5) Physics sanity figure: same real terrain, sun from East vs West (shadows flip) ---
k = 'tycho' if 'tycho' in DEMS else list(DEMS)[0]
dem, px, py, lat0 = DEMS[k]
fig, ax = plt.subplots(2, 3, figsize=(13, 8.4))
for r, (az, el) in enumerate([(90, 30), (270, 30)]):
    im, sh = render_lunar(dem, px, py, az, el)
    ax[r, 0].imshow(im, cmap='gray'); ax[r, 0].set_title(f'{k}: sun az {az} deg, el {el} deg')
    pc, _ = phase_congruency(to_u8(im)); ax[r, 1].imshow(pc_to_u8(pc), cmap='magma'); ax[r, 1].set_title('phase-congruency map')
    ax[r, 2].imshow(sh, cmap='gray'); ax[r, 2].set_title('cast-shadow mask')
    for a in ax[r]: a.axis('off')
plt.tight_layout(); plt.savefig(f'{RES}/fig1_sun_flip_physics.png', dpi=150); plt.show()

In [ ]:
# 6) Load LoFTR (pretrained, zero-shot) -----------------------------------
LOFTR = None
if HAVE_TORCH:
    try:
        from kornia.feature import LoFTR
        LOFTR = LoFTR(pretrained='outdoor').to(DEV).eval(); print('LoFTR loaded')
    except Exception as e:
        print('LoFTR NOT available -> it will be skipped and reported as such:', str(e)[:200])

def loftr_match(a, b, valid_b=None, thr=0.3):
    ta = torch.from_numpy(a)[None, None].float().div(255).to(DEV); tb = torch.from_numpy(b)[None, None].float().div(255).to(DEV)
    with torch.no_grad(): out = LOFTR({'image0': ta, 'image1': tb})
    k0 = out['keypoints0'].cpu().numpy(); k1 = out['keypoints1'].cpu().numpy(); c = out['confidence'].cpu().numpy()
    keep = c > thr
    if valid_b is not None and len(k1):
        ix = np.clip(np.round(k1).astype(int), 0, valid_b.shape[0] - 1); keep &= valid_b[ix[:, 1], ix[:, 0]]
    return k0[keep].astype(np.float32), k1[keep].astype(np.float32)

In [ ]:
# 7) Learned descriptor: trained ON TRAIN REGIONS ONLY, early-stopped on VALIDATION REGIONS ---
P = 48
class PatchNet(nn.Module):
    def __init__(s, dim=128):
        super().__init__()
        def blk(i, o, st): return nn.Sequential(nn.Conv2d(i, o, 3, st, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
        s.f = nn.Sequential(blk(1, 32, 1), blk(32, 32, 1), blk(32, 64, 2), blk(64, 64, 1), blk(64, 128, 2), blk(128, 128, 1),
                            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.1), nn.Linear(128, dim))
    def forward(s, x): return F.normalize(s.f(x), dim=1)

def crop_patches(img, pts, P=P):
    h = P // 2; pad = cv2.copyMakeBorder(img, h, h, h, h, cv2.BORDER_REFLECT)
    out = np.empty((len(pts), P, P), np.uint8)
    for i, (x, y) in enumerate(pts):
        xi = int(np.clip(round(x), 0, img.shape[1] - 1)); yi = int(np.clip(round(y), 0, img.shape[0] - 1)); out[i] = pad[yi:yi + P, xi:xi + P]
    return out

def build_bank(roles, n_suns, rng):
    bank = {}
    for k, (_, _, role) in TILES.items():
        if role not in roles: continue
        dem, px, py, _ = DEMS[k]
        bank[k] = [to_u8(render_lunar(dem, px, py, rng.uniform(0, 360), rng.uniform(8, 45), seed=int(rng.integers(1e9)))[0]) for _ in range(n_suns)]
    return bank

def sample_pairs(bank, n_crops, rng, CW=320, pts_per=24):
    PA, PB = [], []
    keys = list(bank)
    for _ in range(n_crops):
        t = keys[rng.integers(len(keys))]; i, j = rng.choice(len(bank[t]), 2, replace=False)
        x0, y0 = rng.integers(0, SIZE - CW, 2)
        a = bank[t][i][y0:y0 + CW, x0:x0 + CW]; b0 = bank[t][j][y0:y0 + CW, x0:x0 + CW]
        M = cv2.getRotationMatrix2D((CW / 2, CW / 2), rng.uniform(-20, 20), rng.uniform(0.8, 1.25))
        b = cv2.warpAffine(b0, M, (CW, CW)); vb = cv2.erode((cv2.warpAffine(np.ones_like(b0), M, (CW, CW)) > 0).astype(np.uint8), np.ones((3, 3), np.uint8), iterations=P // 2).astype(bool)
        g = cv2.GaussianBlur(a, (0, 0), 1.5); mag = cv2.magnitude(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
        c = cv2.goodFeaturesToTrack(mag, 16, 0.01, 10); c = np.zeros((0, 2), np.float32) if c is None else c.reshape(-1, 2)
        pts = np.vstack([c, rng.uniform(P, CW - P, (pts_per - len(c), 2))]).astype(np.float32)
        mp = np.c_[pts, np.ones(len(pts))] @ M.T
        ok = (mp[:, 0] > P // 2) & (mp[:, 0] < CW - P // 2) & (mp[:, 1] > P // 2) & (mp[:, 1] < CW - P // 2)
        ok &= vb[np.clip(mp[:, 1].astype(int), 0, CW - 1), np.clip(mp[:, 0].astype(int), 0, CW - 1)]
        if ok.sum() == 0: continue
        PA.append(crop_patches(a, pts[ok])); PB.append(crop_patches(b, mp[ok]))
    return np.concatenate(PA), np.concatenate(PB)

def norm_in(u8, aug=False):
    x = u8.float() / 255.
    if aug:
        B = x.shape[0]; g = torch.empty(B, 1, 1, device=x.device).uniform_(0.6, 1.6)
        x = x.clamp(1e-4, 1) ** g * torch.empty(B, 1, 1, device=x.device).uniform_(0.7, 1.3) + torch.randn_like(x) * 0.02
    return ((x - x.mean((1, 2), keepdim=True)) / (x.std((1, 2), keepdim=True) + 1e-3))[:, None]

NET, HIST = None, []
if HAVE_TORCH:
    t0 = time.time(); rng = np.random.default_rng(SEED)
    nsun, ncrop_tr, ncrop_va, EPOCHS = (4, 150, 40, 3) if QUICK else (16, 2200, 350, 25)
    bank_tr = build_bank({'train'}, nsun, rng); bank_va = build_bank({'val'}, max(4, nsun // 2), rng)
    TRA, TRB = sample_pairs(bank_tr, ncrop_tr, rng); VAA, VAB = sample_pairs(bank_va, ncrop_va, rng)
    print(f'train pairs {len(TRA)}  val pairs {len(VAA)}  built in {time.time()-t0:.0f}s')
    TRA, TRB, VAA, VAB = [torch.from_numpy(x).to(DEV) for x in (TRA, TRB, VAA, VAB)]
    NET = PatchNet().to(DEV); opt = torch.optim.AdamW(NET.parameters(), 2e-3, weight_decay=1e-4)
    BS = 512; steps = max(1, len(TRA) // BS); sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=EPOCHS * steps)
    best, best_state = -1, None
    def val_top1():
        NET.eval(); accs = []
        with torch.no_grad():
            for i in range(0, len(VAA) - 63, 512):
                za, zb = NET(norm_in(VAA[i:i + 512])), NET(norm_in(VAB[i:i + 512]))
                accs.append((( za @ zb.T).argmax(1) == torch.arange(len(za), device=DEV)).float().mean().item())
        return float(np.mean(accs))
    for ep in range(EPOCHS):
        NET.train(); perm = torch.randperm(len(TRA), device=DEV); tl = 0
        for s in range(steps):
            idx = perm[s * BS:(s + 1) * BS]
            za, zb = NET(norm_in(TRA[idx], True)), NET(norm_in(TRB[idx], True))
            lg = za @ zb.T / 0.07; lab = torch.arange(len(za), device=DEV)
            loss = (F.cross_entropy(lg, lab) + F.cross_entropy(lg.T, lab)) / 2
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tl += loss.item()
        v = val_top1(); HIST.append(dict(epoch=ep + 1, train_loss=tl / steps, val_top1=v))
        if v > best: best, best_state = v, {k: x.clone() for k, x in NET.state_dict().items()}
        print(f'epoch {ep+1:2d}  train loss {tl/steps:.3f}  val top-1 (unseen regions) {v:.3f}')
    NET.load_state_dict(best_state); NET.eval(); torch.save(best_state, f'{RES}/patchnet_best.pt')
    print(f'best val top-1 = {best:.3f}  | training took {time.time()-t0:.0f}s')
    hd = pd.DataFrame(HIST); fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].plot(hd.epoch, hd.train_loss); ax[0].set_title('train loss (train regions)'); ax[1].plot(hd.epoch, hd.val_top1, color='C1'); ax[1].set_title('val top-1 retrieval (held-out regions)')
    plt.tight_layout(); plt.savefig(f'{RES}/fig2_training_curve.png', dpi=150); plt.show()

In [ ]:
# 8) Unified matcher suite --------------------------------------------------
def detect_pc(pc_u8, valid=None, n=1500):
    m = None if valid is None else (valid.astype(np.uint8) * 255)
    c = cv2.goodFeaturesToTrack(pc_u8, n, 0.005, 7, mask=m)
    return np.zeros((0, 2), np.float32) if c is None else c.reshape(-1, 2).astype(np.float32)

def embed(img, pts):
    if len(pts) == 0: return None
    out = []
    with torch.no_grad():
        for i in range(0, len(pts), 2048):
            p = torch.from_numpy(crop_patches(img, pts[i:i + 2048])).to(DEV); out.append(NET(norm_in(p)))
    return torch.cat(out)

def learned_match(a, b, pcA, pcB, valid_b, ratio=0.92):
    ka, kb = detect_pc(pcA), detect_pc(pcB, valid_b)
    if len(ka) < 5 or len(kb) < 5: return np.zeros((0, 2), np.float32), np.zeros((0, 2), np.float32)
    ea, eb = embed(a, ka), embed(b, kb); S = ea @ eb.T
    v, j = S.topk(2, dim=1); i2 = S.argmax(0)
    d1 = torch.sqrt(torch.clamp(2 - 2 * v[:, 0], min=0)); d2 = torch.sqrt(torch.clamp(2 - 2 * v[:, 1], min=0))
    ok = (d1 < ratio * d2) & (i2[j[:, 0]] == torch.arange(len(ea), device=DEV))
    ok = ok.cpu().numpy(); jj = j[:, 0].cpu().numpy()
    return ka[ok], kb[jj[ok]]

def run_methods(ref, src, valid):
    pa, _ = phase_congruency(ref); pb, _ = phase_congruency(src); A, B = pc_to_u8(pa), pc_to_u8(pb)
    out = {}
    def timed(name, fn):
        t = time.time()
        try: out[name] = (*fn(), time.time() - t)
        except Exception as e: print('  method failed:', name, str(e)[:120]); out[name] = None
    timed('SIFT', lambda: sift_match(ref, src, valid))
    timed('PC-SIFT', lambda: sift_match(A, B, valid))
    if LOFTR is not None: timed('LoFTR (pretrained)', lambda: loftr_match(ref, src, valid))
    if NET is not None: timed('PC + learned descriptor', lambda: learned_match(ref, src, A, B, valid))
    return out

In [ ]:
# 9) Smoke test (1 pair, all methods) — catches crashes before the long run ---
dem, px, py, lat0 = DEMS[[k for k, v in TILES.items() if v[2] == 'test'][0]]
ref, src, Mgt, valid = make_pair(dem, px, py, 90, 30, 270, 25, 8.0, 0.95, seed=1)
for name, r in run_methods(ref, src, valid).items():
    if r is None: continue
    ev = evaluate(r[0], r[1], Mgt, valid, py, SIZE)
    print(f"{name:26s} matches {ev['n_matches']:4d} inliers {ev['n_inliers']:4d} match-precision {ev['match_precision']:.2f} median checkpoint err {ev['med_err_px']:.2f}px  [{r[2]:.1f}s]  gate={gate(ev)}")

In [ ]:
# 10) Main benchmark on HELD-OUT TEST REGIONS (cached in a .jsonl so a disconnect is resumable) ---
GAPS = [0, 60, 120, 180] if QUICK else [0, 30, 60, 90, 120, 150, 180]
ELS = {'high sun (35/30 deg)': (35, 30), 'low sun (12/10 deg)': (12, 10)}
NSEED = 1 if QUICK else 2
LOG = f'{RES}/grid_results.jsonl'
done = set()
if os.path.exists(LOG):
    for l in open(LOG): d = json.loads(l); done.add((d['tile'], d['gap'], d['el_case'], d['seed'], d['method']))
test_tiles = [k for k, v in TILES.items() if v[2] == 'test']
total = len(test_tiles) * len(GAPS) * len(ELS) * NSEED; n = 0; t0 = time.time()
with open(LOG, 'a') as fh:
    for tile in test_tiles:
        dem, px, py, lat0 = DEMS[tile]
        for elc, (el_r, el_s) in ELS.items():
            for gap in GAPS:
                for sd in range(NSEED):
                    n += 1
                    r = np.random.default_rng(stable_seed(tile, gap, elc, sd))
                    rot, sc = r.uniform(-15, 15), r.uniform(0.85, 1.2)
                    if all((tile, gap, elc, sd, m) in done for m in ('SIFT', 'PC-SIFT')) and not QUICK: continue
                    ref, src, Mgt, valid = make_pair(dem, px, py, 90, el_r, (90 + gap) % 360, el_s, rot, sc, seed=1000 + sd)
                    for name, res in run_methods(ref, src, valid).items():
                        if res is None: continue
                        ev = evaluate(res[0], res[1], Mgt, valid, py, SIZE)
                        ev.update(tile=tile, gap=gap, el_case=elc, seed=sd, method=name, gate=gate(ev), time_s=res[2],
                                  gt_success=bool(ev['med_err_px'] < 2.0), rot=rot, scale=sc)
                        fh.write(json.dumps({k: (None if (isinstance(v, float) and not np.isfinite(v)) else v) for k, v in ev.items()}) + '\n')
                    fh.flush()
                    if n % 8 == 0: print(f'{n}/{total} pairs  elapsed {(time.time()-t0)/60:.1f} min')
DF = pd.DataFrame([json.loads(l) for l in open(LOG)])
DF['gt_success'] = DF['gt_success'].astype(bool)
DF['med_err_px'] = DF['med_err_px'].astype(float).fillna(1000).clip(upper=1000)   # failure = 1000 px, never dropped
print(DF.groupby('method').gt_success.mean().round(3))

In [ ]:
# 11) Results figures + honest tables -----------------------------------------
methods = list(DF.method.unique()); cols = {'SIFT': 'tab:red', 'PC-SIFT': 'tab:orange', 'LoFTR (pretrained)': 'tab:blue', 'PC + learned descriptor': 'tab:green'}
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for c, elc in enumerate(ELS):
    d = DF[DF.el_case == elc]
    for m in methods:
        g = d[d.method == m].groupby('gap')
        axes[0, c].plot(g.gt_success.mean().index, g.gt_success.mean().values * 100, 'o-', label=m, color=cols.get(m))
        med = g.med_err_px.median(); axes[1, c].plot(med.index, med.values, 'o-', label=m, color=cols.get(m))
    axes[0, c].set_title(f'Registration success vs sun-azimuth gap - {elc}'); axes[0, c].set_ylabel('% pairs with median checkpoint error < 2 px'); axes[0, c].set_ylim(-3, 103)
    axes[1, c].set_title('median checkpoint error (px), lower is better'); axes[1, c].set_yscale('log'); axes[1, c].set_xlabel('sun azimuth difference (deg)')
axes[0, 0].legend(fontsize=8); [a.grid(alpha=.3) for a in axes.ravel()]
plt.tight_layout(); plt.savefig(f'{RES}/fig3_success_vs_sun_gap.png', dpi=170); plt.show()

tab = DF.pivot_table(index='method', columns='gap', values='gt_success', aggfunc='mean').round(2) * 100
print('SUCCESS RATE (%) by sun gap (all test tiles, both sun-elevation regimes)'); display(tab)
summ = DF.groupby('method').agg(pairs=('gt_success', 'size'), success_pct=('gt_success', lambda x: 100 * x.mean()),
        median_err_px=('med_err_px', 'median'), median_err_ok_only=('med_err_px', lambda x: np.nanmedian(x[x < 2])), median_inliers=('n_inliers', 'median'), match_precision=('match_precision', 'mean'), seconds=('time_s', 'mean')).round(2)
display(summ)

# Is the refusal gate honest? (gate uses only ground-truth-free quantities)
gt = DF.groupby(['method', 'gate']).agg(pairs=('gt_success', 'size'), actually_correct_pct=('gt_success', lambda x: 100 * x.mean())).round(1)
print('REFUSAL GATE RELIABILITY: of pairs the gate labelled X, how many were really correct?'); display(gt)
fig, ax = plt.subplots(figsize=(7, 3.6)); gg = DF.groupby('gate').gt_success.mean() * 100; gg.reindex([x for x in ['SUCCESS', 'DEGRADED', 'REFUSED'] if x in gg.index]).plot.bar(ax=ax, color=['g', 'orange', 'r'][:len(gg)])
ax.set_ylabel('% truly correct (ground truth)'); ax.set_title('Refusal gate vs ground truth (all methods pooled)'); plt.tight_layout(); plt.savefig(f'{RES}/fig4_gate_reliability.png', dpi=150); plt.show()

In [ ]:
# 12) Showcase figure for the slides: match lines + checkerboard (best method on a hard 120 deg pair) ---
def showcase(tile, gap, elc, method, sd=0, fname='fig5_showcase'):
    el_r, el_s = ELS[elc]; dem, px, py, _ = DEMS[tile]
    r = np.random.default_rng(stable_seed(tile, gap, elc, sd)); rot, sc = r.uniform(-15, 15), r.uniform(0.85, 1.2)
    ref, src, Mgt, valid = make_pair(dem, px, py, 90, el_r, (90 + gap) % 360, el_s, rot, sc, seed=1000 + sd)
    res = run_methods(ref, src, valid)[method]; pa, pb = res[0], res[1]; ev = evaluate(pa, pb, Mgt, valid, py, SIZE)
    H, inl = verify(pa, pb)
    if H is None: print('no model'); return
    canvas = np.hstack([cv2.cvtColor(ref, cv2.COLOR_GRAY2RGB), cv2.cvtColor(src, cv2.COLOR_GRAY2RGB)])
    for (x1, y1), (x2, y2), ok in list(zip(pa, pb, inl))[:400]:
        if ok: cv2.line(canvas, (int(x1), int(y1)), (int(x2) + SIZE, int(y2)), (0, 255, 0), 1)
    reg = cv2.warpPerspective(src, np.linalg.inv(H), (SIZE, SIZE)); cb = ref.copy(); T = 64
    for i in range(0, SIZE, T):
        for j in range(0, SIZE, T):
            if (i // T + j // T) % 2: cb[i:i + T, j:j + T] = reg[i:i + T, j:j + T]
    fig, ax = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={'width_ratios': [2, 1]})
    ax[0].imshow(canvas); ax[0].set_title(f'{method}: {ev["n_inliers"]} inliers | sun gap {gap} deg | {tile} ({"SYNTHETIC DEBUG" if USE_SYNTHETIC_DEBUG else "real LOLA DEM"}) | gate {gate(ev)}')
    ax[1].imshow(cb, cmap='gray'); ax[1].set_title(f'checkerboard after registration\nmedian checkpoint error {ev["med_err_px"]:.2f} px vs ground truth')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{RES}/{fname}.png', dpi=170); plt.show()

d = DF[(DF.gap == 120) & (DF.el_case == list(ELS)[0]) & DF.gt_success & (DF.gate == 'SUCCESS')]
if len(d) == 0: d = DF[(DF.gap == 120) & DF.gt_success]
if len(d):
    b = d.sort_values('n_inliers', ascending=False).iloc[0]; showcase(b.tile, int(b.gap), b.el_case, b.method, int(b.seed))
else: print('No successful 120 deg pair in this run - report that honestly (it is a finding).')

In [ ]:
# 13) OPTIONAL: real image pair (e.g. two Chandrayaan-2 OHRC / LROC NAC crops). No ground truth => qualitative only.
# Upload two grayscale PNG/JPG crops of the SAME area (any sun angle). Skip this cell if you have none.
from google.colab import files
up = files.upload()
if len(up) >= 2:
    names_ = list(up)[:2]; a = cv2.imread(names_[0], 0); b = cv2.imread(names_[1], 0)
    S = 640; a = cv2.resize(a, (S, S)); b = cv2.resize(b, (S, S)); vb = np.ones((S, S), bool)
    for name, r in run_methods(a, b, vb).items():
        if r is None: continue
        H, inl = verify(r[0], r[1]); nin = int(inl.sum()); print(f'{name:26s} matches {len(r[0]):4d} inliers {nin:4d} (no ground truth: qualitative)')
    best = 'LoFTR (pretrained)' if LOFTR is not None else 'PC-SIFT'; r = run_methods(a, b, vb)[best]; H, inl = verify(r[0], r[1])
    if H is not None:
        canvas = np.hstack([cv2.cvtColor(a, cv2.COLOR_GRAY2RGB), cv2.cvtColor(b, cv2.COLOR_GRAY2RGB)])
        for (x1, y1), (x2, y2), ok in list(zip(r[0], r[1], inl))[:300]:
            if ok: cv2.line(canvas, (int(x1), int(y1)), (int(x2) + S, int(y2)), (0, 255, 0), 1)
        plt.figure(figsize=(14, 6)); plt.imshow(canvas); plt.axis('off'); plt.title(f'REAL image pair, {best}, {int(inl.sum())} inliers (no ground truth)'); plt.savefig(f'{RES}/fig6_real_pair_qualitative.png', dpi=150); plt.show()

In [ ]:
# 14) Export everything the slides / README / video need -----------------------------
provenance = dict(dem_source=('SYNTHETIC-DEBUG' if USE_SYNTHETIC_DEBUG else LDEM_SRC), dem='LRO LOLA LDEM_64 (~474 m/px)', tiles={k: v for k, v in TILES.items()},
                  size_px=SIZE, gaps=GAPS, el_cases=list(ELS), seeds_per_cell=NSEED, pairs_total=int(len(DF) / max(1, DF.method.nunique())),
                  train_val_history=HIST, best_val_top1=(max(h['val_top1'] for h in HIST) if HIST else None),
                  success_by_method=DF.groupby('method').gt_success.mean().round(3).to_dict(),
                  success_by_method_gap=(DF.pivot_table(index='method', columns='gap', values='gt_success', aggfunc='mean').round(3).to_dict()),
                  gate_reliability=DF.groupby('gate').gt_success.mean().round(3).to_dict(),
                  limits=['coarse 474 m/px DEM, not OHRC', 'similarity transform only', 'no albedo variation', 'no SPICE/PDS4 ingestion', 'learned descriptor is not rotation-invariant beyond +-20 deg'],
                  quick_mode=QUICK, synthetic_debug=USE_SYNTHETIC_DEBUG)
json.dump(provenance, open(f'{RES}/summary.json', 'w'), indent=2, default=str)
DF.to_csv(f'{RES}/results_all_pairs.csv', index=False)
with open(f'{RES}/results_summary.md', 'w') as f:
    f.write('# LunaProof benchmark (real LOLA terrain, held-out regions)\n\n' + tab.to_markdown() + '\n\n' + summ.to_markdown() + '\n\n' + gt.to_markdown())
zp = '/content/lunaproof_results.zip' if os.path.isdir('/content') else f'{RES}.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(RES): z.write(os.path.join(RES, fn), fn)
print('saved:', os.listdir(RES))
try:
    from google.colab import files; files.download(zp)
except Exception: print('zip at', zp)